# Intervention Helpers

Audit coverage for first-class intervention helper constructors and selector wiring.

In [ ]:
# COVERS: torchlens.zero_ablate, torchlens.mean_ablate, torchlens.scale, torchlens.clamp, torchlens.noise
# COVERS: torchlens.func, torchlens.where
import torch
from torch import nn

import torchlens as tl
from torchlens.options import CaptureOptions, VisualizationOptions


class HelperAuditModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)

    def forward(self, x):
        return torch.relu(self.linear(x))


torch.manual_seed(17)
log = tl.log_forward_pass(
    HelperAuditModel().eval(),
    torch.randn(2, 3),
    capture=CaptureOptions(intervention_ready=True, layers_to_save="all", random_seed=0),
    visualization=VisualizationOptions(view="none"),
)

helpers = [
    tl.zero_ablate(),
    tl.mean_ablate(),
    tl.scale(0.5),
    tl.clamp(-1.0, 1.0),
    tl.noise(std=0.01),
]
for helper in helpers:
    assert callable(helper)

log.set(tl.where(tl.func("relu")), tl.zero_ablate(), confirm_mutation=True)
assert log.intervention_spec.targets